In [ ]:
import pandas as pd

# Load dataset
df = pd.read_csv("dirty_cafe_sales.csv")

df1=df.copy()

# Preview data
df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
2,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
3,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
4,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19


In [20]:
#--------------------------------------------------
# CHECKING FOR DUPLICATES & REMOVAL
#--------------------------------------------------

# 1. Identify duplicates
dup_mask = df.duplicated()

print("=== BEFORE: duplicates removed ===")
print('SHAPE:', df.shape, "\n")
print(df.loc[dup_mask])

# 2. Drop duplicates
df = df.drop_duplicates()

print("\n=== AFTER: duplicates removed ===")
#print(df.loc[dup_mask])

print(f"\ndropped {len(dup_mask)-len(df)} \nNumber of rows:{len(df)}")

#===================================
##df=df1.copy()    # rolled back
#-----------------------------------
df1=df.copy() ## update save copy
df.to_csv('cafe_sales.csv', index=True) # Export DataFrame to CSV
#===================================

=== BEFORE: duplicates removed ===
SHAPE: (10002, 8) 

  Transaction ID    Item Quantity Price Per Unit Total Spent Payment Method  \
1    TXN_1961373  Coffee        2            2.0         4.0    Credit Card   
3    TXN_4977031    Cake        4            3.0        12.0           Cash   

   Location Transaction Date  
1  Takeaway       2023-09-08  
3  In-store       2023-05-16  

=== AFTER: duplicates removed ===

dropped 2 
Number of rows:10000


In [21]:
#-----------------------------------
# convert to numeric
#-----------------------------------
print('before convert:------------')
print(df.info())

# convert to float
df["Price Per Unit"] = pd.to_numeric(df["Price Per Unit"], errors="coerce")
df["Total Spent"] = pd.to_numeric(df["Total Spent"], errors="coerce")

# convert to integer
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["Quantity"] = df["Quantity"].astype("Int64")


#-----------------------------------
# convert to date
#-----------------------------------
from datetime import datetime
# convert to date
df["Transaction Date"] = pd.to_datetime(df["Transaction Date"], errors="coerce")

print('after convert:------------')
print(df.info())

#===================================
#df=df1.copy()    # rolled back
#-----------------------------------
df1=df.copy() ## update save copy
df.to_csv('cafe_sales.csv', index=True) # Export DataFrame to CSV
#===================================

before convert:------------
<class 'pandas.core.frame.DataFrame'>
Index: 10000 entries, 0 to 10001
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 703.1+ KB
None
after convert:------------
<class 'pandas.core.frame.DataFrame'>
Index: 10000 entries, 0 to 10001
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    10000 non-null  object        
 1   Item              9667 non-null   object        
 2   Quantity   

In [22]:
all_cols= ["Item", "Quantity", "Price Per Unit", "Total Spent", "Payment Method", "Location", "Transaction Date"]
col_nums= ["Quantity", "Price Per Unit", "Total Spent"]
#-----------------------------------
# Dtype = objects
# Replace "UNKNOWN", "ERROR" & blanks with "DK/NS"
#-----------------------------------

print('before:------------')
df.info()

#cols = ["Item"]
#cols = ["Payment Method"]
#cols = ["Location"]
cols = ["Item", "Payment Method", "Location"]
for col in cols:
    x=df[col].fillna("NULL").value_counts()
#    print(f"--------{col} before replacement\n", x)

    df[col] = df[col].replace(["UNKNOWN", "ERROR"],"DK/NS").fillna("DK/NS")

    x=df[col].value_counts()
#    print(f"-------- {col} after replacement\n", x)
#    print("\n")

print('\nafter:------------')
print(df.info())

#===================================
#df=df1.copy()    # rolled back
#-----------------------------------
df1=df.copy() ## update save copy
df.to_csv('cafe_sales.csv', index=True) # Export DataFrame to CSV
#===================================

before:------------
<class 'pandas.core.frame.DataFrame'>
Index: 10000 entries, 0 to 10001
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    10000 non-null  object        
 1   Item              9667 non-null   object        
 2   Quantity          9521 non-null   Int64         
 3   Price Per Unit    9467 non-null   float64       
 4   Total Spent       9498 non-null   float64       
 5   Payment Method    7421 non-null   object        
 6   Location          6735 non-null   object        
 7   Transaction Date  9540 non-null   datetime64[ns]
dtypes: Int64(1), datetime64[ns](1), float64(2), object(4)
memory usage: 712.9+ KB

after:------------
<class 'pandas.core.frame.DataFrame'>
Index: 10000 entries, 0 to 10001
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    

In [23]:
#-----------------------------------
# INITITAL RUN: Total = Quantity * Price
#-----------------------------------

print('before:------------')
print(df.info())

#--------------------------------------------------
# FIX Total Spent = Quantity * Price Per Unit
#--------------------------------------------------
mask_tq = (
    df["Total Spent"].isna() &
    df["Quantity"].notna() & df["Price Per Unit"].notna()
)
#print("=== BEFORE ===")
#print(df.loc[mask_tq, ["Quantity", "Price Per Unit", "Total Spent"]])
df.loc[mask_tq, "Total Spent"] = (df.loc[mask_tq, "Quantity"] * df.loc[mask_tq, "Price Per Unit"])
#print("\n=== AFTER ===")
#print(df.loc[mask_tq, ["Quantity", "Price Per Unit", "Total Spent"]])

#--------------------------------------------------
# FIX Quantity = Total Spent / Price Per Unit
#--------------------------------------------------
mask_q = (
    df["Quantity"].isna() &
    df["Total Spent"].notna() & df["Price Per Unit"].notna()
)
#print("=== BEFORE ===")
#print(df.loc[mask_q, ["Quantity", "Price Per Unit", "Total Spent"]])
df.loc[mask_q, "Quantity"] = (df.loc[mask_q, "Total Spent"] / df.loc[mask_q, "Price Per Unit"])
#print("\n=== AFTER ===")
#print(df.loc[mask_q, ["Quantity", "Price Per Unit", "Total Spent"]])

#--------------------------------------------------
# FIX Price Per Unit = Total Spent / Quantity
#--------------------------------------------------
mask_p = (
    df["Price Per Unit"].isna() &
    df["Total Spent"].notna() & df["Quantity"].notna()
)
#print("=== BEFORE ===")
#print(df.loc[mask_p, ["Quantity", "Price Per Unit", "Total Spent"]])
df.loc[mask_p, "Price Per Unit"] = (df.loc[mask_p, "Total Spent"] / df.loc[mask_p, "Quantity"])
#print("\n=== AFTER ===")
#print(df.loc[mask_p, ["Quantity", "Price Per Unit", "Total Spent"]])

print('after:------------')
print(df.info())

#===================================
#df=df1.copy()    # rolled back
#-----------------------------------
df1=df.copy() ## update save copy
df.to_csv('cafe_sales.csv', index=True) # Export DataFrame to CSV
#===================================

before:------------
<class 'pandas.core.frame.DataFrame'>
Index: 10000 entries, 0 to 10001
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    10000 non-null  object        
 1   Item              10000 non-null  object        
 2   Quantity          9521 non-null   Int64         
 3   Price Per Unit    9467 non-null   float64       
 4   Total Spent       9498 non-null   float64       
 5   Payment Method    10000 non-null  object        
 6   Location          10000 non-null  object        
 7   Transaction Date  9540 non-null   datetime64[ns]
dtypes: Int64(1), datetime64[ns](1), float64(2), object(4)
memory usage: 712.9+ KB
None
after:------------
<class 'pandas.core.frame.DataFrame'>
Index: 10000 entries, 0 to 10001
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID

In [24]:
for col in all_cols:
    if col != "Price Per Unit":
#        print(f"\n=== Price Per Unit by {col} ===")
#        print(df.groupby(col)["Price Per Unit"].value_counts())
        pass
#----------- only item may make sense here & re-tabulate below

# Price count
price_counts = (
    df.groupby("Price Per Unit")["Item"]
      .value_counts(dropna=False)   # count items within each price
      .sort_index()                 # sort prices ascending
)
#print(f"\n=== Items by Price Per Unit (inc DK/NS) ===\n{price_counts}")

#--------------------------------------------------
# From price_counts: see rule dictionary
#--------------------------------------------------
rules = {
    1.0: "Cookie",
    1.5: "Tea",
    2.0: "Coffee",
    5.0: "Salad"
}
print("\nFix: Items VS. Price")
for price, item_name in rules.items():
    # Mask for rows to be fixed
    mask_valid          = (df["Item"] == item_name) & (df["Price Per Unit"] == price)
    mask_missing_items  = (df["Item"] == "DK/NS") & (df["Price Per Unit"] == price)
    mask_missing_price = (df["Item"] == item_name) & (df["Price Per Unit"].isna())

    # BEFORE: missing items
    before_missing_items = mask_missing_items.sum()
    before_count_valid = mask_valid.sum()
    print(f"\nPrice {price:.2f} → {item_name}={before_count_valid}")
    print(f"\tBefore: {before_missing_items} missing items rows to fix")

    # Apply the fix
    df.loc[mask_missing_items, "Item"] = item_name
    mask_valid          = (df["Item"] == item_name) & (df["Price Per Unit"] == price)

    # AFTER: verify changes
    after_count_valid = mask_valid.sum()
    print(f"\tAfter: {after_count_valid} rows now set to {item_name}")

    # REVERSE --------------
    # BEFORE: missing prices
    before_missing_prices = mask_missing_price.sum()
    print(f"Item= {item_name} → Price {price:.2f}")
    print(f"\tBefore: {before_missing_prices} rows with missing price")

    # Apply fix
    df.loc[mask_missing_price, "Price Per Unit"] = price

    # AFTER
    after_fixed = ((df["Item"] == item_name) & (df["Price Per Unit"] == price)).sum()
    print(f"\tAfter:  {after_fixed} rows now set to price {price:.2f}")


# Price Per Unit=3.0, Item either Cake OR Juice
# Price Per Unit=4.0, Item either Sandwich OR Smoothie
rules1 = {
    3.0: "Cake",
    3.0: "Juice",
    4.0: "Sandwich",
    4.0: "Smoothie",
}
for price, item_name in rules1.items():
    # Mask for rows to be fixed
    mask_valid          = (df["Item"] == item_name) & (df["Price Per Unit"] == price)
    mask_missing_price = (df["Item"] == item_name) & (df["Price Per Unit"].isna())

    # if cake its 3.0, but if its a 3.0 unsure if its a cake or juice
    print('\n')
    
    # REVERSE --------------
    # BEFORE: missing prices
    before_missing_prices = mask_missing_price.sum()
    print(f"Item= {item_name} → Price {price:.2f}")
    print(f"\tBefore: {before_missing_prices} rows with missing price")

    # Apply fix
    df.loc[mask_missing_price, "Price Per Unit"] = price

    # AFTER
    after_fixed = ((df["Item"] == item_name) & (df["Price Per Unit"] == price)).sum()
    print(f"\tAfter:  {after_fixed} rows now set to price {price:.2f}")

#===================================
#df=df1.copy()    # rolled back
#-----------------------------------
df1=df.copy() ## update save copy
df.to_csv('cafe_sales.csv', index=True) # Export DataFrame to CSV
#===================================


Fix: Items VS. Price

Price 1.00 → Cookie=1086
	Before: 121 missing items rows to fix
	After: 1207 rows now set to Cookie
Item= Cookie → Price 1.00
	Before: 6 rows with missing price
	After:  1213 rows now set to price 1.00

Price 1.50 → Tea=1082
	Before: 118 missing items rows to fix
	After: 1200 rows now set to Tea
Item= Tea → Price 1.50
	Before: 7 rows with missing price
	After:  1207 rows now set to price 1.50

Price 2.00 → Coffee=1163
	Before: 126 missing items rows to fix
	After: 1289 rows now set to Coffee
Item= Coffee → Price 2.00
	Before: 2 rows with missing price
	After:  1291 rows now set to price 2.00

Price 5.00 → Salad=1146
	Before: 124 missing items rows to fix
	After: 1270 rows now set to Salad
Item= Salad → Price 5.00
	Before: 2 rows with missing price
	After:  1272 rows now set to price 5.00


Item= Juice → Price 3.00
	Before: 1 rows with missing price
	After:  1171 rows now set to price 3.00


Item= Smoothie → Price 4.00
	Before: 5 rows with missing price
	After:  1

In [25]:
#-----------------------------------
# SECOND RUN: Total = Quantity * Price
#-----------------------------------

print('before:------------')
print(df.info())

#--------------------------------------------------
# FIX Total Spent = Quantity * Price Per Unit
#--------------------------------------------------
mask_tq = (
    df["Total Spent"].isna() &
    df["Quantity"].notna() & df["Price Per Unit"].notna()
)
#print("=== BEFORE ===")
#print(df.loc[mask_tq, ["Quantity", "Price Per Unit", "Total Spent"]])
df.loc[mask_tq, "Total Spent"] = (df.loc[mask_tq, "Quantity"] * df.loc[mask_tq, "Price Per Unit"])
#print("\n=== AFTER ===")
#print(df.loc[mask_tq, ["Quantity", "Price Per Unit", "Total Spent"]])

#--------------------------------------------------
# FIX Quantity = Total Spent / Price Per Unit
#--------------------------------------------------
mask_q = (
    df["Quantity"].isna() &
    df["Total Spent"].notna() & df["Price Per Unit"].notna()
)
#print("=== BEFORE ===")
#print(df.loc[mask_q, ["Quantity", "Price Per Unit", "Total Spent"]])
df.loc[mask_q, "Quantity"] = (df.loc[mask_q, "Total Spent"] / df.loc[mask_q, "Price Per Unit"])
#print("\n=== AFTER ===")
#print(df.loc[mask_q, ["Quantity", "Price Per Unit", "Total Spent"]])

#--------------------------------------------------
# FIX Price Per Unit = Total Spent / Quantity
#--------------------------------------------------
mask_p = (
    df["Price Per Unit"].isna() &
    df["Total Spent"].notna() & df["Quantity"].notna()
)
#print("=== BEFORE ===")
#print(df.loc[mask_p, ["Quantity", "Price Per Unit", "Total Spent"]])
df.loc[mask_p, "Price Per Unit"] = (df.loc[mask_p, "Total Spent"] / df.loc[mask_p, "Quantity"])
#print("\n=== AFTER ===")
#print(df.loc[mask_p, ["Quantity", "Price Per Unit", "Total Spent"]])

print('after:------------')
print(df.info())

#===================================
#df=df1.copy()    # rolled back
#-----------------------------------
df1=df.copy() ## update save copy
df.to_csv('cafe_sales.csv', index=True) # Export DataFrame to CSV
#===================================

before:------------
<class 'pandas.core.frame.DataFrame'>
Index: 10000 entries, 0 to 10001
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    10000 non-null  object        
 1   Item              10000 non-null  object        
 2   Quantity          9962 non-null   Int64         
 3   Price Per Unit    9985 non-null   float64       
 4   Total Spent       9960 non-null   float64       
 5   Payment Method    10000 non-null  object        
 6   Location          10000 non-null  object        
 7   Transaction Date  9540 non-null   datetime64[ns]
dtypes: Int64(1), datetime64[ns](1), float64(2), object(4)
memory usage: 712.9+ KB
None
after:------------
<class 'pandas.core.frame.DataFrame'>
Index: 10000 entries, 0 to 10001
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID

In [26]:
#--------------------------------------------------
# DROP those with 2 or more nulls
#--------------------------------------------------
print('before:------------')
print(df.info())

mask_2nulls = df[["Quantity", "Price Per Unit", "Total Spent"]].isna().sum(axis=1) >= 2

rows_dropped = mask_2nulls.sum()
df = df[~mask_2nulls]
print(f"\nRows dropped due to 2+ nulls: {rows_dropped}\n")

print('after:------------')
print(df.info())
#===================================
#df=df1.copy()    # rolled back
#-----------------------------------
df1=df.copy() ## update save copy
df.to_csv('cafe_sales.csv', index=True) # Export DataFrame to CSV
#===================================

before:------------


<class 'pandas.core.frame.DataFrame'>
Index: 10000 entries, 0 to 10001
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    10000 non-null  object        
 1   Item              10000 non-null  object        
 2   Quantity          9973 non-null   Int64         
 3   Price Per Unit    9985 non-null   float64       
 4   Total Spent       9972 non-null   float64       
 5   Payment Method    10000 non-null  object        
 6   Location          10000 non-null  object        
 7   Transaction Date  9540 non-null   datetime64[ns]
dtypes: Int64(1), datetime64[ns](1), float64(2), object(4)
memory usage: 712.9+ KB
None

Rows dropped due to 2+ nulls: 35

after:------------
<class 'pandas.core.frame.DataFrame'>
Index: 9965 entries, 0 to 10001
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   

In [27]:
#------------------------------------
# Summary
#------------------------------------
print("DK/NS counts and %")
shape=df.shape[0]
print(f"no of row: {shape} \n-------------")

for col in all_cols:
    dk_count = (df[col] == "DK/NS").sum()
    nan_count = df[col].isna().sum()
    total = dk_count + nan_count
    percent = total / shape * 100
    print(f"{col:<20} {total:<5} ({percent:5.2f}%)")

#===================================
#df=df1.copy()    # rolled back
#-----------------------------------
df1=df.copy() ## update save copy
df.to_csv('cafe_sales.csv', index=True) # Export DataFrame to CSV
#===================================

DK/NS counts and %
no of row: 9965 
-------------
Item                 474   ( 4.76%)
Quantity             0     ( 0.00%)
Price Per Unit       0     ( 0.00%)
Total Spent          0     ( 0.00%)
Payment Method       3164  (31.75%)
Location             3951  (39.65%)
Transaction Date     460   ( 4.62%)
